<a href="https://colab.research.google.com/github/sabascodes/Logistics-project/blob/main/DelayModel_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             mean_absolute_error, r2_score)

In [ ]:
from google.colab import files

uploaded = files.upload()
name = list(uploaded.keys())[0]

if name.endswith(".xlsx"):
    pd.read_excel(name).to_csv("delay_model_dataset.csv", index=False)
    DATA_PATH = "delay_model_dataset.csv"
else:
    DATA_PATH = name

MODEL_PATH = "delay_model.joblib"
RANDOM_STATE = 42


Saving delay_model_dataset.csv to delay_model_dataset (1).csv


In [ ]:
CATEGORICAL = ["origin_city", "destination_city", "transportation_mode",
               "season", "weather_condition", "traffic_level", "priority_level"]

NUMERIC = ["distance_km", "shipment_weight_kg", "order_lead_time_hours",
           "carrier_rating", "vehicle_capacity_utilization",
           "planned_transit_hours", "month", "day_of_week"]

FEATURES = CATEGORICAL + NUMERIC


In [ ]:
def add_date_features(df):
    """Turn shipment_date into month and day_of_week."""
    df = df.copy()
    dates = pd.to_datetime(df["shipment_date"])
    df["month"] = dates.dt.month
    df["day_of_week"] = dates.dt.dayofweek
    return df


def make_preprocessor():
    """Text columns -> one-hot (0/1 columns); number columns -> scaled."""
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("num", StandardScaler(), NUMERIC),
    ])


In [ ]:
# 2. TRAINING
# ---------------------------------------------------------------------------
def train(data_path=DATA_PATH, model_path=MODEL_PATH):
    df = add_date_features(pd.read_csv(data_path))
    X = df[FEATURES]
    y_class = df["is_delayed"]      # 0 / 1
    y_hours = df["delay_hours"]     # can be negative = arrived early

    # 80% to learn from, 20% kept hidden to test honestly
    X_tr, X_te, yc_tr, yc_te, yh_tr, yh_te = train_test_split(
        X, y_class, y_hours, test_size=0.2, stratify=y_class,
        random_state=RANDOM_STATE)

    print(f"Rows: {len(df)} | train: {len(X_tr)} | test: {len(X_te)}")
    print(f"Delayed shipments: {y_class.mean():.1%}\n")

In [ ]:
df = add_date_features(pd.read_csv(DATA_PATH))
X = df[FEATURES]
y_class = df["is_delayed"]      # 0 / 1
y_hours = df["delay_hours"]     # can be negative = arrived early

# 80% to learn from, 20% kept hidden to test honestly
X_tr, X_te, yc_tr, yc_te, yh_tr, yh_te = train_test_split(
    X, y_class, y_hours, test_size=0.2, stratify=y_class,
    random_state=RANDOM_STATE)

print(f"Rows: {len(df)} | train: {len(X_tr)} | test: {len(X_te)}")
print(f"Delayed shipments: {y_class.mean():.1%}\n")

# ---- A) Classifier: compare a simple baseline vs Random Forest ----
candidates = {
        "Logistic Regression (baseline)": LogisticRegression(max_iter=2000),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5,
            random_state=RANDOM_STATE, n_jobs=-1),
 }
print("=== DELAY CLASSIFIER (is the shipment delayed?) ===")
fitted = {}
for name, model in candidates.items():
        pipe = Pipeline([("prep", make_preprocessor()), ("model", model)])
        pipe.fit(X_tr, yc_tr)
        proba = pipe.predict_proba(X_te)[:, 1]
        pred = (proba >= 0.5).astype(int)
        fitted[name] = pipe
        print(f"{name:32s} accuracy={accuracy_score(yc_te, pred):.3f}  "
              f"precision={precision_score(yc_te, pred):.3f}  "
              f"recall={recall_score(yc_te, pred):.3f}  "
              f"F1={f1_score(yc_te, pred):.3f}  "
              f"AUC={roc_auc_score(yc_te, proba):.3f}")

clf = fitted["Random Forest"]

proba = clf.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(yc_te, pred).ravel()
print(f"\nConfusion matrix (Random Forest): "
          f"correct on-time={tn}, false alarms={fp}, missed delays={fn}, "
          f"caught delays={tp}")

Rows: 6000 | train: 4800 | test: 1200
Delayed shipments: 47.7%

=== DELAY CLASSIFIER (is the shipment delayed?) ===
Logistic Regression (baseline)   accuracy=0.801  precision=0.786  recall=0.801  F1=0.793  AUC=0.876
Random Forest                    accuracy=0.833  precision=0.835  recall=0.812  F1=0.823  AUC=0.906

Confusion matrix (Random Forest): correct on-time=535, false alarms=92, missed delays=108, caught delays=465


In [ ]:
    # 5-fold cross-validation = a more trustworthy score than one split
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_auc = cross_val_score(
        Pipeline([("prep", make_preprocessor()),
                  ("model", RandomForestClassifier(
                      n_estimators=300, min_samples_leaf=5,
                      random_state=RANDOM_STATE, n_jobs=-1))]),
        X, y_class, cv=cv, scoring="roc_auc")
    print(f"5-fold cross-validated AUC: {cv_auc.mean():.3f} (+/- {cv_auc.std():.3f})")

5-fold cross-validated AUC: 0.899 (+/- 0.006)


In [ ]:
# ---- B) Regressor: how many hours late? ----
reg = Pipeline([("prep", make_preprocessor()),
                    ("model", RandomForestRegressor(
                        n_estimators=300, min_samples_leaf=5,
                        random_state=RANDOM_STATE, n_jobs=-1))])

reg.fit(X_tr, yh_tr)
h_pred = reg.predict(X_te)
baseline_mae = mean_absolute_error(yh_te, np.full(len(yh_te), yh_tr.mean()))
print("\n=== DELAY-HOURS REGRESSOR (how many hours late?) ===")
print(f"MAE = {mean_absolute_error(yh_te, h_pred):.2f} hours "
          f"(guessing the average every time gives {baseline_mae:.2f})")
print(f"R^2 = {r2_score(yh_te, h_pred):.3f}")



=== DELAY-HOURS REGRESSOR (how many hours late?) ===
MAE = 1.32 hours (guessing the average every time gives 2.44)
R^2 = 0.682


In [ ]:
# ---- C) What matters most? ----
names = clf.named_steps["prep"].get_feature_names_out()
imp = pd.Series(clf.named_steps["model"].feature_importances_, index=names)
imp.index = [n.split("__", 1)[1] for n in imp.index]
print("\nTop 10 factors driving delay (Random Forest):")
print(imp.sort_values(ascending=False).head(10).round(3).to_string())



Top 10 factors driving delay (Random Forest):
weather_condition_Clear         0.134
weather_condition_Sandstorm     0.092
transportation_mode_truck       0.087
order_lead_time_hours           0.087
planned_transit_hours           0.081
traffic_level_Low               0.075
vehicle_capacity_utilization    0.069
carrier_rating                  0.053
traffic_level_High              0.045
shipment_weight_kg              0.041


In [ ]:
# ---- D) Sanity check on the new city: Hail ----
test_df = X_te.copy()
test_df["y"] = yc_te
test_df["pred"] = pred
hail = test_df[test_df["destination_city"] == "Hail"]
print(f"\nHail-only test rows: {len(hail)} | "
          f"accuracy = {(hail['y'] == hail['pred']).mean():.3f}")


Hail-only test rows: 264 | accuracy = 0.841


In [ ]:
 # ---- E) Retrain on ALL data for the final saved model ----
clf.fit(X, y_class)
reg.fit(X, y_hours)
joblib.dump({"classifier": clf, "regressor": reg, "features": FEATURES}, MODEL_PATH)
print(f"\nSaved final models to {MODEL_PATH}")



Saved final models to delay_model.joblib


In [ ]:
# 3. PREDICTION  (this is what the AI agent will call)
# ---------------------------------------------------------------------------
_bundle = None


def predict_delay(shipment: dict, model_path=MODEL_PATH):
    """
    Input : one shipment as a dict (see example below)
    Output: dict the LLM agent can read directly

    'shipment_date' can be given as 'YYYY-MM-DD'.
    """
    global _bundle
    if _bundle is None:
        _bundle = joblib.load(model_path)

    row = add_date_features(pd.DataFrame([shipment]))[_bundle["features"]]
    prob = float(_bundle["classifier"].predict_proba(row)[0, 1])
    hours = float(_bundle["regressor"].predict(row)[0])
    planned = float(shipment["planned_transit_hours"])

    if prob < 0.35:
        risk = "Low"
    elif prob < 0.65:
        risk = "Medium"
    else:
        risk = "High"

    return {
        "delay_probability": round(prob, 3),
        "risk_level": risk,
        "expected_delay_hours": round(max(hours, 0), 2),
        "planned_transit_hours": planned,
        "expected_transit_hours": round(planned + max(hours, 0), 2),
    }


if __name__ == "__main__":
    # Google Colab: pick delay_model_dataset.csv from your computer when prompted
    from google.colab import files
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
    train(path)

    example = {
        "origin_city": "Riyadh", "destination_city": "Hail",
        "transportation_mode": "truck", "distance_km": 627,
        "shipment_date": "2025-03-10", "season": "Ramadan",
        "weather_condition": "Sandstorm", "traffic_level": "High",
        "priority_level": "Urgent", "shipment_weight_kg": 800,
        "order_lead_time_hours": 12, "carrier_rating": 3.2,
        "vehicle_capacity_utilization": 0.9, "planned_transit_hours": 11.5,
    }
    print("\nExample prediction (Riyadh -> Hail, truck, sandstorm, high traffic):")
    print(predict_delay(example))

Saving delay_model_dataset.csv to delay_model_dataset (2).csv
Rows: 6000 | train: 4800 | test: 1200
Delayed shipments: 47.7%


Example prediction (Riyadh -> Hail, truck, sandstorm, high traffic):
{'delay_probability': 0.976, 'risk_level': 'High', 'expected_delay_hours': 11.93, 'planned_transit_hours': 11.5, 'expected_transit_hours': 23.43}
